List of the functions implemented in this notebook for the implementation of the "No-U-turn" algorithm for Hamiltonian Monte Carlo sampling:
1. L example (gasussian log-density) -> OK
2. Leapfrog -> Ok
3. pseudo_energy -> OK
4. FindReasonableEpsilon -> OK
5. BuildTree + auxiliary functions 

The functions are implemented in JAX in order to exploit its capabilitied to speed up the computations.

In [1]:
# Usare numba con @njit

In [2]:
import math 
import numpy as np
import jax.random as jrnd
import jax.numpy as jnp
from jax import grad, jit, jax, lax
from numpy.typing import NDArray
from typing import Callable, NamedTuple, Tuple


In [3]:
#https://docs.jax.dev/en/latest/
# jax example for AD
import jax.numpy as jnp
from jax import grad

def f(x):
    return jnp.sin(x)

df = grad(f)
print(df(0.))

1.0


In [4]:
# Define the pseudo log-potential
# Gaussian log-density
@jax.jit
def L(theta: jnp.ndarray) -> jnp.float32:
    assert theta.ndim == 1
    return -0.5 * jnp.vdot(theta, theta) 


In [5]:
# 2. JAX automatic differentiation
grad_L = jax.jit(grad(L))

In [6]:
# 3. pass the values not the function
# Leapfrog integrator
@jax.jit
def Leapfrog(
    theta: jnp.ndarray,
    r: jnp.ndarray,
    epsilon: float) -> Tuple[jnp.ndarray, jnp.ndarray]:

    # Shape discipline
    assert theta.ndim == 1
    assert r.ndim == 1

    # Make epsilon a JAX scalar to avoid recompilation
    epsilon = jnp.asarray(epsilon)

    # half-step momentum update
    grad0 = grad_L(theta)
    r_tilde = r + 0.5 * epsilon * grad0

    # full-step position update
    theta_tilde = theta + epsilon * r_tilde

    # second half-step momentum update
    grad1 = grad_L(theta_tilde)
    r_tilde = r_tilde + 0.5 * epsilon * grad1

    return theta_tilde, r_tilde

In [7]:
# This calculate a value proportional to log(p(r,theta)) that is the hamiltonian value at r,theta;
# Working in log-space provide more stability and is useful in order to avoid overflows

@jax.jit
def log_p(theta: NDArray[float], r: NDArray[float]) -> float:

    # Shape discipline
    assert theta.ndim == 1
    assert r.ndim == 1

    # "potential" + "kinetic"
    E = L(theta) - ( 0.5 * jnp.vdot(r, r) )
    return E

In [8]:
# Alghorithm 4 of the article

def FindReasonableEpsilon(theta: NDArray[float], key=jnp.array([0, 0])) -> float:
    assert theta.ndim == 1

    # sample momentum r ~ N(0, I)
    key, subkey = jrnd.split(key)
    r = jrnd.normal(subkey, shape=theta.shape)

    # initial epsilon
    epsilon = 1.0

    # compute initial log joint density
    E0 = log_p(theta, r)

    # one leapfrog step
    theta_prime, r_prime = Leapfrog(theta, r, epsilon)
    E1 = log_p(theta_prime, r_prime)

    # log acceptance probability
    log_accept = E1 - E0

    # direction: +1 (increase) or -1 (decrease)
    a = jnp.where(log_accept > jnp.log(0.5), 1.0, -1.0)

    # -------------------------------
    # cond_fun: while loop consition
    # -------------------------------
    def cond_fun(state):
        eps, log_acc = state
        return (a * log_acc) > (-a * jnp.log(2.0))

    # --------------------
    # body_fun: loop body
    # --------------------
    def body_fun(state):
        eps, _ = state
        eps = eps * (2.0 ** a)
        theta_p, r_p = Leapfrog(theta, r, eps)
        E1 = log_p(theta_p, r_p)
        log_acc = E1 - E0
        return (eps, log_acc)

    # while loop in JAX
    epsilon, _ = jax.lax.while_loop(cond_fun, body_fun, (epsilon, log_accept))

    return epsilon


In [9]:
# input: [theta, r, u, v, j, epsilon, theta_0, r_0]
# theta,r -> coordinates
# u -> slice variable
# v -> {+1,-1} direction
# j -> number of previous doublings

class Root(NamedTuple):
    theta: jnp.ndarray
    r: jnp.ndarray
    u: float
    v: int
    j: int
    epsilon: float
    theta_0: jnp.ndarray
    r_0: jnp.ndarray


# Define a class to mange the tree variables
class Tree(NamedTuple):
    theta_minus: jnp.ndarray
    r_minus: jnp.ndarray
    theta_plus: jnp.ndarray
    r_plus: jnp.ndarray
    theta_prime: jnp.ndarray
    n_prime: jnp.ndarray
    s_prime: jnp.ndarray
    alpha_prime: jnp.ndarray
    n_a_prime: jnp.ndarray


#@jax.jit
def build_tree_base(root: Root) -> Tree:
    theta, r = root.theta, root.r
    u, v = root.u, root.v
    eps = root.epsilon
    theta0, r0 = root.theta_0, root.r_0

    theta_p, r_p = Leapfrog(theta, r, v * eps)

    logu = jnp.log(u)
    lp = log_p(theta_p, r_p)

    n_p = (logu <= lp).astype(jnp.int32)
    s_p = (logu < (1000.0 + lp)).astype(jnp.int32)

    delta = (
        L(theta_p) - 0.5 * jnp.dot(r_p, r_p)
        - L(theta0) + 0.5 * jnp.dot(r0, r0)
    )
    alpha_p = jnp.minimum(1.0, jnp.exp(delta))
    n_a_p = jnp.array(1, dtype=jnp.int32)

    return Tree(
        theta_minus=theta_p,
        r_minus=r_p,
        theta_plus=theta_p,
        r_plus=r_p,
        theta_prime=theta_p,
        n_prime=n_p,
        s_prime=s_p,
        alpha_prime=alpha_p,
        n_a_prime=n_a_p,
    )

# function to verify if the stop condition is reached
@jax.jit
def stop_criterion(theta_minus, theta_plus, r_minus, r_plus) -> jnp.int32:
    delta = theta_plus - theta_minus
    cond = (jnp.dot(delta, r_minus) >= 0) & (jnp.dot(delta, r_plus) >= 0)
    return cond.astype(jnp.int32)
    
'''
# Function in the algorithm 6 of the article
def BuildTree(root: Root, key):
    """
    Pure Jupyter implementation 
    using an object of Root class as input.
    """

    # Base case: j == 0
    def base_case(_):
        return build_tree_base(root), key

    # Recursive case: j > 0
    def recursive_case(_):

        # --- 1. Build the left subtree ---
        left_tree, key_left = BuildTree(
            root._replace(j=root.j - 1),
            key
        )

        # --- 2. If it is all ok, build the right subtree ---
        def build_right(_):
            # Adjust the root
            new_root = root._replace(
                theta = left_tree.theta_minus if root.v == -1 else left_tree.theta_plus,
                r     = left_tree.r_minus     if root.v == -1 else left_tree.r_plus,
                j     = root.j - 1
            )
            return BuildTree(new_root, key_left)

        right_tree, key_right = lax.cond(
            left_tree.s_prime == 1,
            build_right,
            lambda _: (left_tree, key_left),
            operand=None
        )

        # --- 3. Combine the trees ---
        theta_minus = left_tree.theta_minus if root.v == -1 else right_tree.theta_minus
        r_minus     = left_tree.r_minus     if root.v == -1 else right_tree.r_minus
        theta_plus  = right_tree.theta_plus if root.v == 1  else left_tree.theta_plus
        r_plus      = right_tree.r_plus     if root.v == 1  else left_tree.r_plus

        # --- 4. Choose the values for theta_prime ---
        key_right, sub = jax.random.split(key_right)
        #p = right_tree.n_prime / (left_tree.n_prime + right_tree.n_prime + 1e-12)
        total = left_tree.n_prime + right_tree.n_prime
        p = jnp.where(total > 0, right_tree.n_prime / total, 0.5)
        choose = jax.random.bernoulli(sub, p)
        theta_prime = jnp.where(choose, right_tree.theta_prime, left_tree.theta_prime)

        # --- 5. Update the statistics ---
        n_prime = left_tree.n_prime + right_tree.n_prime
        s_prime = left_tree.s_prime * right_tree.s_prime * stop_criterion(theta_minus, theta_plus, r_minus, r_plus)
        alpha_prime = left_tree.alpha_prime + right_tree.alpha_prime
        n_a_prime = left_tree.n_a_prime + right_tree.n_a_prime

        merged = Tree(
            theta_minus=theta_minus,
            r_minus=r_minus,
            theta_plus=theta_plus,
            r_plus=r_plus,
            theta_prime=theta_prime,
            n_prime=n_prime,
            s_prime=s_prime,
            alpha_prime=alpha_prime,
            n_a_prime=n_a_prime,
        )

        return merged, key_right

    # --- Select the base case or the recursive one ---
    return lax.cond(root.j == 0, base_case, recursive_case, operand=None)
'''

def BuildTree(root: Root, key):
    """
    Pure-Python recursive BuildTree (no JAX control flow).
    """

    # Base case: j == 0
    if root.j == 0:
        return build_tree_base(root), key

    # Recursive case: j > 0

    # --- 1. Left subtree ---
    left_root = root._replace(j=root.j - 1)
    left_tree, key_left = BuildTree(left_root, key)

    # --- 2. Right subtree only if still valid ---
    if left_tree.s_prime == 1:
        if root.v == -1:
            new_root = root._replace(
                theta=left_tree.theta_minus,
                r=left_tree.r_minus,
                j=root.j - 1,
            )
        else:
            new_root = root._replace(
                theta=left_tree.theta_plus,
                r=left_tree.r_plus,
                j=root.j - 1,
            )
        right_tree, key_right = BuildTree(new_root, key_left)
    else:
        right_tree, key_right = left_tree, key_left

    # --- 3. Merge bounds ---
    theta_minus = left_tree.theta_minus if root.v == -1 else right_tree.theta_minus
    r_minus     = left_tree.r_minus     if root.v == -1 else right_tree.r_minus
    theta_plus  = right_tree.theta_plus if root.v == 1  else left_tree.theta_plus
    r_plus      = right_tree.r_plus     if root.v == 1  else left_tree.r_plus

    # --- 4. Choose theta_prime ---
    key_right, sub = jrnd.split(key_right)
    total_n = left_tree.n_prime + right_tree.n_prime
    p = jnp.where(total_n > 0, right_tree.n_prime / total_n, 0.5)
    choose = jrnd.bernoulli(sub, p)
    theta_prime = jnp.where(choose, right_tree.theta_prime, left_tree.theta_prime)

    # --- 5. Stats ---
    n_prime = left_tree.n_prime + right_tree.n_prime
    s_prime = left_tree.s_prime * right_tree.s_prime * stop_criterion(
        theta_minus, theta_plus, r_minus, r_plus
    )
    alpha_prime = left_tree.alpha_prime + right_tree.alpha_prime
    n_a_prime = left_tree.n_a_prime + right_tree.n_a_prime

    merged = Tree(
        theta_minus=theta_minus,
        r_minus=r_minus,
        theta_plus=theta_plus,
        r_plus=r_plus,
        theta_prime=theta_prime,
        n_prime=n_prime,
        s_prime=s_prime,
        alpha_prime=alpha_prime,
        n_a_prime=n_a_prime,
    )

    return merged, key_right
